# LCProp PR

Start with a fresh Colab CPU runtime and the reviewed LCProp wheel with its Colab extra. Verify the wheel checksum and restart after installation before running the display cell. Only the interface Run button starts propagation.

For the optional PR T4 preview, select a Colab T4 runtime and follow the setup guide. In a fresh kernel, change `ENABLE_COLAB_T4` below to `True`. Leave it `False` for the CPU default.


## Install once in a fresh runtime

Upload the staged release wheel into `/content` using the Colab Files pane. Run the install cell, restart the session, then start at the verification cell. The install cell also repairs missing Jedi in Colab images where IPython requires it.

In [ ]:
import hashlib
from pathlib import Path
import subprocess
import sys

wheel = Path("/content/lcprop-0.5.0rc1-py3-none-any.whl")
expected_sha256 = "fa660391769ad93270b55674d60a35a6e4b779a865c6d091418166db68ac6ceb"
if hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("The uploaded wheel does not match the pinned release candidate.")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(wheel)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", f"lcprop[colab] @ {wheel.as_uri()}", "jedi"],
    check=True,
)
print("Restart the session and continue at Verify installation.")

## Verify installation

Run this cell after restarting. Then run the interface cell below.

In [ ]:
from importlib import metadata
import json
import subprocess
import sys

expected_sha256 = "fa660391769ad93270b55674d60a35a6e4b779a865c6d091418166db68ac6ceb"
dist = metadata.distribution("lcprop")
origin = json.loads(dist.read_text("direct_url.json") or "{}")
installed_sha256 = origin.get("archive_info", {}).get("hashes", {}).get("sha256")
if dist.version != "0.5.0rc1" or installed_sha256 != expected_sha256:
    raise RuntimeError("Install the pinned wheel and restart this session first.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("Verified LCProp", dist.version, installed_sha256)

In [ ]:
from google.colab import output
from IPython import get_ipython
from IPython.display import clear_output
from lcprop.notebook import PRNotebook

ENABLE_COLAB_T4 = False

if "lcprop_pr" not in globals():
    output.enable_custom_widget_manager()
    get_ipython().run_line_magic("matplotlib", "widget")
    if ENABLE_COLAB_T4:
        from lcprop.notebook.batch_device import DeviceUnavailable, probe_cupy

        def probe_colab_t4(spec):
            device = probe_cupy(spec)
            if device.name.strip() not in {"Tesla T4", "NVIDIA T4", "T4"}:
                raise DeviceUnavailable("The Colab PR GPU preview requires a T4 runtime.")
            return device

        lcprop_pr = PRNotebook(gpu_probe=probe_colab_t4)
    else:
        lcprop_pr = PRNotebook()
    _lcprop_pr_t4_enabled = ENABLE_COLAB_T4
elif lcprop_pr.session.disposed or globals().get("_lcprop_pr_t4_enabled", False) != ENABLE_COLAB_T4:
    raise RuntimeError("Save needed files and restart the kernel before changing controller setup.")
clear_output(wait=True)
lcprop_pr.display_in_colab()
